# 意图识别

先使用大模型生成数据，然后直接训练BERT分类

In [146]:
from __future__ import annotations

import json
import random
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm
import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE = {DEVICE}")
PATH = '/bohr/train-z7v5/v1/'
BASE_MODEL_NAME = "bert-base-chinese"
# MODEL_DIR = PATH + "models--bert-base-chinese/8f23c25b06e129b6c986331a13d8d025a92cf0ea"
MODEL_DIR = PATH + "bert-base-chinese"
TRAIN_PATH = Path(PATH + "/train.jsonl")


MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 5e-5
SEED = 42


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


set_seed(SEED)

In [142]:
INTENT_LABELS = [
    "游戏技巧", "游戏角色信息", "周边饭馆", "查找高端酒店", "查找地址",
    "健康知识", "查找医疗信息", "美容化妆技巧", "美食烹饪技巧", "软件开发问题",
    "软件使用问题", "查找小说剧情", "查找小说角色信息", "查找营业时间",
    "法律法条解释", "法律问题咨询",
]
LABEL_TO_ID = {l: i for i, l in enumerate(INTENT_LABELS)}
ID_TO_LABEL = {i: l for i, l in enumerate(INTENT_LABELS)}

In [144]:
import json
from typing import Any, List, Dict

# 假设 LABEL_TO_ID 是一个字典，映射意图标签到对应的 ID
LABEL_TO_ID = {
    "游戏技巧": 0,
    "游戏角色信息": 1,
    "周边饭馆": 2,
    "查找高端酒店": 3,
    "查找地址": 4,
    "健康知识": 5,
    "查找医疗信息": 6,
    "美容化妆技巧": 7,
    "美食烹饪技巧": 8,
    "软件开发问题": 9,
    "软件使用问题": 10,
    "查找小说剧情": 11,
    "查找小说角色信息": 12,
    "查找营业时间": 13,
    "法律法条解释": 14,
    "法律问题咨询": 15
}

def extract_user_utterance(text: str) -> str:
    user_utts = []
    for line in text.strip().split("\n"):
        if line.strip().startswith("usr:"):
            user_utts.append(line.strip()[4:].strip())
    return user_utts[-1] if user_utts else ""

def preprocess_input_text(text: Any) -> str:
    if text is None:
        return ""
    s = str(text).strip()
    if not s:
        return ""
    if "usr:" in s:
        u = extract_user_utterance(s)
        if u:
            return u
    return s

def load_jsonl(path):
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]

def add_training_data(new_samples: List[Dict[str, str]], existing_data: List[Dict[str, Any]]):
    for sample in new_samples:
        text = preprocess_input_text(sample.get("input"))
        lab = str(sample.get("label", "")).strip()
        if text and lab in LABEL_TO_ID:
            existing_data.append({"text": text, "label_id": LABEL_TO_ID[lab]})

# 加载现有训练数据
train_records = load_jsonl(TRAIN_PATH)
train_rows = []
for r in train_records:
    text = preprocess_input_text(r.get("input"))
    lab = str(r.get("label", "")).strip()
    if text and lab in LABEL_TO_ID:
        train_rows.append({"text": text, "label_id": LABEL_TO_ID[lab]})

# 新增样本数据
new_samples = [
    # 游戏技巧
    {"input": "如何在LOL中快速上分？", "label": "游戏技巧"},
    {"input": "DNF中哪个职业最强？", "label": "游戏技巧"},
    {"input": "如何在PUBG中提高生存率？", "label": "游戏技巧"},
    {"input": "王者荣耀的打野技巧有哪些？", "label": "游戏技巧"},
    {"input": "如何在Minecraft中建造房屋？", "label": "游戏技巧"},

    # 游戏角色信息
    {"input": "英雄联盟中的阿狸有什么技能？", "label": "游戏角色信息"},
    {"input": "DNF中的剑魂角色特点是什么？", "label": "游戏角色信息"},
    {"input": "王者荣耀的貂蝉背景故事是什么？", "label": "游戏角色信息"},
    {"input": "PUBG中的角色有什么不同？", "label": "游戏角色信息"},
    {"input": "Minecraft中的村民有什么作用？", "label": "游戏角色信息"},

    # 周边饭馆
    {"input": "我想找一家评分高的意大利餐厅", "label": "周边饭馆"},
    {"input": "附近有没有好吃的烧烤店？", "label": "周边饭馆"},
    {"input": "请推荐一家适合家庭聚餐的餐馆", "label": "周边饭馆"},
    {"input": "我想找一家有素食选项的餐厅", "label": "周边饭馆"},
    {"input": "附近的快餐店有哪些？", "label": "周边饭馆"},

    # 查找高端酒店
    {"input": "请推荐一家五星级酒店", "label": "查找高端酒店"},
    {"input": "我想找一家靠近机场的高档酒店", "label": "查找高端酒店"},
    {"input": "有哪些适合商务出差的酒店？", "label": "查找高端酒店"},
    {"input": "请告诉我这附近的豪华酒店", "label": "查找高端酒店"},
    {"input": "我想预定一家有游泳池的酒店", "label": "查找高端酒店"},

    # 查找地址
    {"input": "请告诉我最近的地铁站在哪里？", "label": "查找地址"},
    {"input": "我想知道这家店的具体地址", "label": "查找地址"},
    {"input": "如何到达最近的医院？", "label": "查找地址"},
    {"input": "请给我这家餐厅的地址", "label": "查找地址"},
    {"input": "我想找去图书馆的路线", "label": "查找地址"},

    # 健康知识
    {"input": "每天喝水的好处有哪些？", "label": "健康知识"},
    {"input": "如何保持心理健康？", "label": "健康知识"},
    {"input": "吃水果的最佳时间是什么？", "label": "健康知识"},
    {"input": "怎样才能提高免疫力？", "label": "健康知识"},
    {"input": "运动对身体有哪些好处？", "label": "健康知识"},

    # 查找医疗信息
    {"input": "请问这附近的医院有哪些？", "label": "查找医疗信息"},
    {"input": "我想了解一下疫苗接种的信息", "label": "查找医疗信息"},
    {"input": "如何预约医生的门诊？", "label": "查找医疗信息"},
    {"input": "请告诉我最近的药店在哪里", "label": "查找医疗信息"},
    {"input": "我想知道体检的流程", "label": "查找医疗信息"},

    # 美容化妆技巧
    {"input": "如何选择适合自己的粉底液？", "label": "美容化妆技巧"},
    {"input": "请问如何画眼线？", "label": "美容化妆技巧"},
    {"input": "护肤的正确步骤是什么？", "label": "美容化妆技巧"},
    {"input": "如何选择适合自己的口红颜色？", "label": "美容化妆技巧"},
    {"input": "请问如何卸妆才不会伤害皮肤？", "label": "美容化妆技巧"},

    # 美食烹饪技巧
    {"input": "如何制作经典的意大利面？", "label": "美食烹饪技巧"},
    {"input": "请教我怎么做家常豆腐？", "label": "美食烹饪技巧"},
    {"input": "我想学做蛋糕，有什么简单的食谱？", "label": "美食烹饪技巧"},
    {"input": "如何制作健康的沙拉？", "label": "美食烹饪技巧"},
    {"input": "请问怎么做红烧肉？", "label": "美食烹饪技巧"},

    # 软件开发问题
    {"input": "如何使用Python进行数据分析？", "label": "软件开发问题"},
    {"input": "请问如何调试JavaScript代码？", "label": "软件开发问题"},
    {"input": "我想学习前端开发，有什么推荐的资源？", "label": "软件开发问题"},
    {"input": "如何优化SQL查询性能？", "label": "软件开发问题"},
    {"input": "请问如何使用Git进行版本控制？", "label": "软件开发问题"},

    # 软件使用问题
    {"input": "如何在Word中插入页码？", "label": "软件使用问题"},
    {"input": "请问如何使用Excel制作图表？", "label": "软件使用问题"},
    {"input": "如何在Photoshop中调整图片大小？", "label": "软件使用问题"},
    {"input": "我想知道如何在PowerPoint中添加动画效果？", "label": "软件使用问题"},
    {"input": "如何在Zoom中设置虚拟背景？", "label": "软件使用问题"},

    # 查找小说剧情
    {"input": "请告诉我《红楼梦》的主要情节", "label": "查找小说剧情"},
    {"input": "《三国演义》的故事背景是什么？", "label": "查找小说剧情"},
    {"input": "《西游记》中孙悟空的经历有哪些？", "label": "查找小说剧情"},
    {"input": "《哈利·波特》系列的主要角色有哪些？", "label": "查找小说剧情"},
    {"input": "《围城》的主题是什么？", "label": "查找小说剧情"},

    # 查找小说角色信息
    {"input": "请问《红楼梦》中的林黛玉是什么样的人？", "label": "查找小说角色信息"},
    {"input": "《三国演义》中的诸葛亮有什么特点？", "label": "查找小说角色信息"},
    {"input": "《西游记》中的唐僧有什么性格特点？", "label": "查找小说角色信息"},
    {"input": "《哈利·波特》中的伏地魔是谁？", "label": "查找小说角色信息"},
    {"input": "《围城》中的方鸿渐是个怎样的人？", "label": "查找小说角色信息"},

    # 查找营业时间
    {"input": "这家餐厅的营业时间是什么？", "label": "查找营业时间"},
    {"input": "请问附近的超市几点关门？", "label": "查找营业时间"},
    {"input": "我想知道这家咖啡馆的营业时间", "label": "查找营业时间"},
    {"input": "请告诉我这家书店的营业时间", "label": "查找营业时间"},
    {"input": "这家健身房的开放时间是几点到几点？", "label": "查找营业时间"},

    # 法律法条解释
    {"input": "请问《民法典》中关于合同的规定是什么？", "label": "法律法条解释"},
    {"input": "如何理解《刑法》中关于盗窃的条款？", "label": "法律法条解释"},
    {"input": "《劳动法》中关于加班的规定是什么？", "label": "法律法条解释"},
    {"input": "《婚姻法》中关于离婚的条款有哪些？", "label": "法律法条解释"},
    {"input": "请问《知识产权法》是如何保护著作权的？", "label": "法律法条解释"},

    # 法律问题咨询
    {"input": "我想咨询一下如何处理劳动争议？", "label": "法律问题咨询"},
    {"input": "请问我该如何申请离婚？", "label": "法律问题咨询"},
    {"input": "如果我被诬告，我该怎么办？", "label": "法律问题咨询"},
    {"input": "如何维护我的消费者权益？", "label": "法律问题咨询"},
    {"input": "请问我可以如何申请专利？", "label": "法律问题咨询"},

    {"input": "在绝地求生中，如何选择降落地点？", "label": "游戏技巧"},
    {"input": "如何在Minecraft中制作火把？", "label": "游戏技巧"},
    {"input": "在GTA5中，如何快速赚取游戏币？", "label": "游戏技巧"},
    {"input": "如何在魔兽世界中组队打副本？", "label": "游戏技巧"},
    {"input": "在守望先锋中，如何选择合适的英雄？", "label": "游戏技巧"},

    # 游戏角色信息
    {"input": "请介绍一下《英雄联盟》中的德莱文。", "label": "游戏角色信息"},
    {"input": "《魔兽世界》中的法师有什么技能？", "label": "游戏角色信息"},
    {"input": "《最终幻想》系列中的克劳德是个怎样的角色？", "label": "游戏角色信息"},
    {"input": "《塞尔达传说》中的林克有什么特别之处？", "label": "游戏角色信息"},
    {"input": "《生化危机》中的艾达王是个怎样的人物？", "label": "游戏角色信息"},

    # 周边饭馆
    {"input": "我想找一家适合商务宴请的餐厅。", "label": "周边饭馆"},
    {"input": "附近有没有提供自助餐的地方？", "label": "周边饭馆"},
    {"input": "请推荐一家有户外座位的咖啡馆。", "label": "周边饭馆"},
    {"input": "我想找一家有儿童游乐区的餐厅。", "label": "周边饭馆"},
    {"input": "这附近的海鲜餐厅有哪些？", "label": "周边饭馆"},

    # 查找高端酒店
    {"input": "请问有哪些适合蜜月旅行的酒店？", "label": "查找高端酒店"},
    {"input": "我想找一家有SPA服务的酒店。", "label": "查找高端酒店"},
    {"input": "请推荐一家有会议室的高档酒店。", "label": "查找高端酒店"},
    {"input": "我想找一家可以带宠物的酒店。", "label": "查找高端酒店"},
    {"input": "这附近的精品酒店有哪些？", "label": "查找高端酒店"},

    # 查找地址
    {"input": "请问最近的健身房在哪里？", "label": "查找地址"},
    {"input": "我想知道这家书店的具体位置。", "label": "查找地址"},
    {"input": "如何到达最近的公园？", "label": "查找地址"},
    {"input": "请给我这家药店的地址。", "label": "查找地址"},
    {"input": "我想找去博物馆的最佳路线。", "label": "查找地址"},

    # 健康知识
    {"input": "如何有效减肥？", "label": "健康知识"},
    {"input": "请问怎样才能改善睡眠质量？", "label": "健康知识"},
    {"input": "有哪些食物可以帮助提高记忆力？", "label": "健康知识"},
    {"input": "如何预防感冒？", "label": "健康知识"},
    {"input": "请问怎样才能保持良好的心理状态？", "label": "健康知识"},

    # 查找医疗信息
    {"input": "请问这附近的牙科诊所有哪些？", "label": "查找医疗信息"},
    {"input": "我想了解一下体检的项目。", "label": "查找医疗信息"},
    {"input": "如何找到专业的心理咨询师？", "label": "查找医疗信息"},
    {"input": "请告诉我最近的急救中心在哪里。", "label": "查找医疗信息"},
    {"input": "我想咨询一下疫苗接种的时间安排。", "label": "查找医疗信息"},

    # 美容化妆技巧（续）
    {"input": "如何正确使用面膜？", "label": "美容化妆技巧"},
    {"input": "请问如何选择适合自己的香水？", "label": "美容化妆技巧"},
    {"input": "如何让睫毛膏不结块？", "label": "美容化妆技巧"},
    {"input": "如何选择合适的眉笔颜色？", "label": "美容化妆技巧"},
    {"input": "请问如何打造自然的腮红效果？", "label": "美容化妆技巧"},

    # 美食烹饪技巧
    {"input": "如何制作简单的意大利面？", "label": "美食烹饪技巧"},
    {"input": "请问怎么做健康的早餐？", "label": "美食烹饪技巧"},
    {"input": "如何制作美味的牛肉炖菜？", "label": "美食烹饪技巧"},
    {"input": "我想学做寿司，有什么简单的步骤？", "label": "美食烹饪技巧"},
    {"input": "如何制作自制披萨？", "label": "美食烹饪技巧"},

    # 软件开发问题
    {"input": "如何在Python中处理异常？", "label": "软件开发问题"},
    {"input": "请问如何使用Java进行多线程编程？", "label": "软件开发问题"},
    {"input": "如何在前端使用React构建组件？", "label": "软件开发问题"},
    {"input": "请问如何优化网站的加载速度？", "label": "软件开发问题"},
    {"input": "如何在Node.js中处理文件上传？", "label": "软件开发问题"},

    # 软件使用问题
    {"input": "如何在Excel中使用数据透视表？", "label": "软件使用问题"},
    {"input": "请问如何在Word中插入图表？", "label": "软件使用问题"},
    {"input": "如何在PowerPoint中添加视频？", "label": "软件使用问题"},
    {"input": "我想知道如何在Photoshop中使用图层？", "label": "软件使用问题"},
    {"input": "如何在Zoom中共享屏幕？", "label": "软件使用问题"},

    # 查找小说剧情
    {"input": "请问《白夜行》的主要情节是什么？", "label": "查找小说剧情"},
    {"input": "《追风筝的人》的故事背景是什么？", "label": "查找小说剧情"},
    {"input": "《傲慢与偏见》的主要角色有哪些？", "label": "查找小说剧情"},
    {"input": "《百年孤独》的主题是什么？", "label": "查找小说剧情"},
    {"input": "请告诉我《小王子》的故事大意。", "label": "查找小说剧情"},

    # 查找小说角色信息
    {"input": "请问《白夜行》中的角色有哪些？", "label": "查找小说角色信息"},
    {"input": "《追风筝的人》中的哈桑是个怎样的人？", "label": "查找小说角色信息"},
    {"input": "《傲慢与偏见》中的达西有什么特点？", "label": "查找小说角色信息"},
    {"input": "《百年孤独》中的乌尔苏拉是个怎样的角色？", "label": "查找小说角色信息"},
    {"input": "请问《小王子》中的小王子有什么特别之处？", "label": "查找小说角色信息"},

    {"input": "请问这家咖啡馆的营业时间是什么？", "label": "查找营业时间"},
    {"input": "我想知道这家书店几点关门？", "label": "查找营业时间"},
    {"input": "这家餐厅的营业时间是几点到几点？", "label": "查找营业时间"},
    {"input": "请问附近的超市什么时候开门？", "label": "查找营业时间"},
    {"input": "我想了解这家健身房的开放时间。", "label": "查找营业时间"},

     {"input": "在《绝地求生》中，如何选择最佳的降落地点？", "label": "游戏技巧"},
    
    # 游戏角色信息
    {"input": "请介绍一下《英雄联盟》中的劫的技能和特点。", "label": "游戏角色信息"},
    
    # 周边饭馆
    {"input": "我想找一家适合家庭聚餐的餐厅，推荐一下。", "label": "周边饭馆"},
    
    # 查找高端酒店
    {"input": "请问这附近有哪些高档酒店适合度假？", "label": "查找高端酒店"},
    
    # 查找地址
    {"input": "最近的地铁站在哪里？", "label": "查找地址"},
    
    # 健康知识
    {"input": "如何保持良好的心理健康？", "label": "健康知识"},
    
    # 查找医疗信息
    {"input": "请问这附近的医院有哪些？", "label": "查找医疗信息"},
    
    # 美容化妆技巧
    {"input": "如何选择适合自己的护肤品？", "label": "美容化妆技巧"},
    
    # 美食烹饪技巧
    {"input": "如何制作简单的家常菜？", "label": "美食烹饪技巧"},
    
    # 软件开发问题
    {"input": "如何在 Python 中实现多线程？", "label": "软件开发问题"},
    
    # 软件使用问题
    {"input": "如何在 Excel 中制作图表？", "label": "软件使用问题"},
    
    # 查找小说剧情
    {"input": "《红楼梦》的主要剧情是什么？", "label": "查找小说剧情"},
    
    # 查找小说角色信息
    {"input": "请介绍一下《三国演义》中的刘备。", "label": "查找小说角色信息"},
    
    # 查找营业时间
    {"input": "这家咖啡店的营业时间是什么？", "label": "查找营业时间"},
    
    # 法律法条解释
    {"input": "《民法典》中关于合同的规定是什么？", "label": "法律法条解释"},
    
    # 法律问题咨询
    {"input": "如果我在工作中受伤，应该如何索赔？", "label": "法律问题咨询"},
    
    # 游戏技巧
    {"input": "在《Minecraft》中，如何快速获取资源？", "label": "游戏技巧"},
    
    # 游戏角色信息
    {"input": "请介绍一下《守望先锋》中的源氏。", "label": "游戏角色信息"},
    
    # 周边饭馆
    {"input": "我想找一家适合约会的餐厅，推荐一下。", "label": "周边饭馆"},
    
    # 查找高端酒店
    {"input": "请问这附近有哪些适合商务出差的酒店？", "label": "查找高端酒店"},
    
    # 查找地址
    {"input": "最近的健身房在哪里？", "label": "查找地址"},
    
    # 健康知识
    {"input": "如何有效减肥？", "label": "健康知识"},
    
    # 查找医疗信息
    {"input": "请问这附近的药店营业时间是什么？", "label": "查找医疗信息"},
    
    # 美容化妆技巧
    {"input": "如何选择适合自己的发型？", "label": "美容化妆技巧"},
    
    # 美食烹饪技巧
    {"input": "如何制作简单的蛋糕？", "label": "美食烹饪技巧"},
    
    # 软件开发问题
    {"input": "如何在 Java 中实现接口？", "label": "软件开发问题"},
    
    # 软件使用问题
    {"input": "如何在 Photoshop 中调整图片大小？", "label": "软件使用问题"},
    
    # 查找小说剧情
    {"input": "请问《哈利·波特》的主要剧情是什么？", "label": "查找小说剧情"},
    
    # 查找小说角色信息
    {"input": "请介绍一下《西游记》中的唐僧。", "label": "查找小说角色信息"},
        # 查找营业时间
    {"input": "请问这家书店的营业时间是什么？", "label": "查找营业时间"},
    
    # 法律法条解释
    {"input": "《刑法》中关于盗窃罪的规定是什么？", "label": "法律法条解释"},
    
    # 法律问题咨询
    {"input": "如果我被误解为犯罪，应该如何维护自己的权益？", "label": "法律问题咨询"},
    
    # 游戏技巧
    {"input": "在《GTA V》中，如何快速赚取游戏币？", "label": "游戏技巧"},
    
    # 游戏角色信息
    {"input": "请介绍一下《英雄联盟》中的德莱厄斯。", "label": "游戏角色信息"},
    
    # 周边饭馆
    {"input": "我想找一家适合朋友聚会的餐厅，推荐一下。", "label": "周边饭馆"},
    
    # 查找高端酒店
    {"input": "请问这附近有哪些适合家庭度假的酒店？", "label": "查找高端酒店"},
    
    # 查找地址
    {"input": "最近的公园在哪里？", "label": "查找地址"},
    
    # 健康知识
    {"input": "如何提高身体的免疫力？", "label": "健康知识"},
    
    # 查找医疗信息
    {"input": "请问这附近的妇科医院有哪些？", "label": "查找医疗信息"},
    
    # 美容化妆技巧
    {"input": "如何选择适合自己的化妆品？", "label": "美容化妆技巧"},
    
    # 美食烹饪技巧
    {"input": "如何制作简单的炒面？", "label": "美食烹饪技巧"},
    
    # 软件开发问题
    {"input": "如何在 C# 中处理异常？", "label": "软件开发问题"},
    
    # 软件使用问题
    {"input": "如何在 PowerPoint 中插入视频？", "label": "软件使用问题"},
    
    # 查找小说剧情
    {"input": "《西游记》的主要剧情是什么？", "label": "查找小说剧情"},
    
    # 查找小说角色信息
    {"input": "请介绍一下《红楼梦》中的贾宝玉。", "label": "查找小说角色信息"},
    
    # 查找营业时间
    {"input": "这家餐厅的营业时间是什么？", "label": "查找营业时间"},
    
    # 法律法条解释
    {"input": "《劳动法》中关于休假的规定是什么？", "label": "法律法条解释"},
    
    # 法律问题咨询
    {"input": "如果我在商场购物时受伤，应该如何索赔？", "label": "法律问题咨询"},
    
    # 游戏技巧
    {"input": "在《堡垒之夜》中，如何提高自己的建造速度？", "label": "游戏技巧"},
    
    # 游戏角色信息
    {"input": "请介绍一下《DOTA2》中的水人。", "label": "游戏角色信息"},
    
    # 周边饭馆
    {"input": "我想找一家适合商务宴请的餐厅，推荐一下。", "label": "周边饭馆"},
    
    # 查找高端酒店
    {"input": "请问这附近有哪些适合蜜月旅行的酒店？", "label": "查找高端酒店"},
    
    # 查找地址
    {"input": "最近的图书馆在哪里？", "label": "查找地址"},
    
    # 健康知识
    {"input": "如何保持健康的饮食习惯？", "label": "健康知识"},
    
    # 查找医疗信息
    {"input": "请问这附近的中医院有哪些？", "label": "查找医疗信息"},
    
    # 美容化妆技巧
    {"input": "如何画一个自然的妆容？", "label": "美容化妆技巧"},
    
    # 美食烹饪技巧
    {"input": "如何制作简单的披萨？", "label": "美食烹饪技巧"},
    
    # 软件开发问题
    {"input": "如何在 Ruby 中实现面向对象编程？", "label": "软件开发问题"},
    

    
]



# 添加新样本到现有训练数据
add_training_data(new_samples, train_rows)






print(f"训练样本: {len(train_rows)}")
print(train_rows)


In [145]:
from transformers import BertTokenizer, BertModel
class IntentDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len, with_label=True):
        self.rows = rows; self.tokenizer = tokenizer; self.max_len = max_len; self.with_label = with_label

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        enc = self.tokenizer(row["text"], add_special_tokens=True, truncation=True,
                             max_length=self.max_len, padding="max_length", return_tensors="pt")
        item = {"input_ids": enc["input_ids"].flatten(), "attention_mask": enc["attention_mask"].flatten()}
        if self.with_label:
            item["labels"] = torch.tensor(row["label_id"], dtype=torch.long)
        return item


class BertIntentClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        if isinstance(out, tuple):
            pooled = out[0][:, 0]
        else:
            pooled = getattr(out, "pooler_output", None)
            if pooled is None:
                pooled = out.last_hidden_state[:, 0]
        return self.classifier(self.dropout(pooled))


tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
model = BertIntentClassifier(MODEL_DIR, len(INTENT_LABELS)).to(DEVICE)

train_loader = DataLoader(IntentDataset(train_rows, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
optim = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = max(1, len(train_loader) * EPOCHS)
warmup_steps = max(1, int(total_steps * 0.1))
sched = get_linear_schedule_with_warmup(optim, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

for ep in range(EPOCHS):
    print(f"epoch {ep+1}/{EPOCHS}")
    model.train()
    for batch in tqdm(train_loader, desc="train", leave=False):
        optim.zero_grad()
        x = batch["input_ids"].to(DEVICE); m = batch["attention_mask"].to(DEVICE); y = batch["labels"].to(DEVICE)
        loss = nn.CrossEntropyLoss()(model(x, m), y)
        loss.backward(); optim.step(); sched.step()
print("training done.")

In [5]:
import os

if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"  
else:
    print("Baseline运行时，因为无法读取测试集，所以会有此条报错，属于正常现象")  
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")

TEST_A_PATH = Path(DATA_PATH + "/val.jsonl")
TEST_B_PATH = Path(DATA_PATH + "/test.jsonl")


@torch.no_grad()
def predict_to_file(test_path, out_path):
    model.eval()
    records = load_jsonl(test_path)
    rows = [{"text": preprocess_input_text(r.get("input")) or ""} for r in records]
    loader = DataLoader(IntentDataset(rows, tokenizer, MAX_LEN, with_label=False), batch_size=BATCH_SIZE*4, shuffle=False)
    preds = []
    for batch in tqdm(loader, desc="predict", leave=False):
        x = batch["input_ids"].to(DEVICE); m = batch["attention_mask"].to(DEVICE)
        preds.extend(torch.argmax(model(x, m), dim=1).cpu().tolist())
    assert len(preds) == len(records)
    with Path(out_path).open("w", encoding="utf-8") as fh:
        for p in preds:
            fh.write(json.dumps({"label": ID_TO_LABEL[p]}, ensure_ascii=False) + "\n")
    print(f"wrote {len(preds)} -> {out_path}")


predict_to_file(TEST_A_PATH, Path("submission_val.jsonl"))
predict_to_file(TEST_B_PATH, Path("submission_test.jsonl"))

In [9]:
SUBMISSION_ZIP = Path("submission.zip")
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission_val.jsonl")
    zf.write("submission_test.jsonl")
print(f"wrote {SUBMISSION_ZIP} ({SUBMISSION_ZIP.stat().st_size} bytes)")

In [13]:
'''

import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 扩展用户主目录路径
model_path = os.path.expanduser("~/Qwen3-4B-Instruct")

# 检测可用设备
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"正在使用设备: {device}")

# 加载分词器和模型（使用自动推断的精度，若GPU可用则自动使用bfloat16）
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

# 定义对话消息（遵循Qwen的chat模板）
messages = [
    {"role": "system", "content": "你是一个有用的助手。"},
    {"role": "user", "content": "你好，请介绍一下你自己。"}
]

# 应用聊天模板生成输入
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 对输入进行编码并移至模型设备
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# 生成回复参数（可根据需要调整）
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    repetition_penalty=1.05
)

# 仅提取生成的部分（去除原始输入）
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# 解码回复
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("\n模型回复：")
print(response)
'''